# NTPDase1 MD Trajectory Analysis

Companion notebook to: *Conserved Hinge-Region Mutations Produce Geometry-Specific Structural Effects in Leishmania NTPDase1: A Cross-Species Computational Target Validation Study*

**Author:** Dan Szolc | University of Salford

## Overview

This notebook analyses the four MD trajectories produced by `NTPDase1_MD_Simulation.ipynb`. It calculates:

- Backbone RMSD for whole protein and defined regions
- Per-residue Cα RMSF
- Statistical significance (block-averaged t-tests for RMSD, paired t-tests for RMSF)
- Percentage change (wild-type vs mutant)

## Regions Analysed

| Region | Residues |
|--------|----------|
| N-terminal tail | 1-39 |
| Conserved hinge | 40-51 |
| N-terminal + hinge | 1-55 |
| Catalytic core | 52-425 |

## Equilibration

The first 2.5 ns of each trajectory is discarded as equilibration. All quantitative analysis is performed on the remaining 22.5 ns production window (2.5-25 ns).

## 1. Setup and Drive Mount

In [ ]:
!pip install mdanalysis mdtraj matplotlib numpy pandas scipy -q

import mdtraj as md
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

output_folder = '/content/drive/MyDrive/NTPDase1_MD/results'
pdb_folder = '/content/drive/MyDrive/NTPDase1_MD'

print("Analysis tools ready")

## 2. Load Trajectories

Trajectories are loaded from Google Drive. Water and ions are stripped after loading to reduce memory usage for downstream analysis. Requires High RAM runtime.

In [ ]:
# L. infantum
print("Loading L. infantum WT trajectory...")
wt_traj = md.load(
    f"{output_folder}/Linfantum_WT_trajectory.dcd",
    top=f"{output_folder}/Linfantum_WT_final.pdb"
)
print(f"WT: {wt_traj.n_frames} frames, {wt_traj.n_atoms} atoms")

print("Loading L. infantum MUT trajectory...")
mut_traj = md.load(
    f"{output_folder}/Linfantum_MUT_trajectory.dcd",
    top=f"{output_folder}/Linfantum_MUT_final.pdb"
)
print(f"MUT: {mut_traj.n_frames} frames, {mut_traj.n_atoms} atoms")

In [ ]:
# L. major
print("Loading L. major WT trajectory...")
lmaj_wt = md.load(
    f"{output_folder}/Lmajor_WT_trajectory.dcd",
    top=f"{output_folder}/Lmajor_WT_final.pdb"
)
print(f"WT: {lmaj_wt.n_frames} frames, {lmaj_wt.n_atoms} atoms")

print("Loading L. major MUT trajectory...")
lmaj_mut = md.load(
    f"{output_folder}/Lmajor_MUT_trajectory.dcd",
    top=f"{output_folder}/Lmajor_MUT_final.pdb"
)
print(f"MUT: {lmaj_mut.n_frames} frames, {lmaj_mut.n_atoms} atoms")

## 3. Strip Water and Extract Protein Atoms

Only protein atoms are needed for RMSD/RMSF. Removing water reduces memory usage by ~50x.

In [ ]:
# Extract protein-only trajectories
wt_protein = wt_traj.atom_slice(wt_traj.topology.select('protein'))
mut_protein = mut_traj.atom_slice(mut_traj.topology.select('protein'))
lmaj_wt_protein = lmaj_wt.atom_slice(lmaj_wt.topology.select('protein'))
lmaj_mut_protein = lmaj_mut.atom_slice(lmaj_mut.topology.select('protein'))

print(f"L. infantum WT protein:  {wt_protein.n_frames} frames, {wt_protein.n_atoms} atoms")
print(f"L. infantum MUT protein: {mut_protein.n_frames} frames, {mut_protein.n_atoms} atoms")
print(f"L. major WT protein:     {lmaj_wt_protein.n_frames} frames, {lmaj_wt_protein.n_atoms} atoms")
print(f"L. major MUT protein:    {lmaj_mut_protein.n_frames} frames, {lmaj_mut_protein.n_atoms} atoms")

# Free memory
del wt_traj, mut_traj, lmaj_wt, lmaj_mut

## 4. Discard First 2.5 ns as Equilibration

Frame counts depend on `report_interval` used during simulation. For 100 ps intervals (used for MUT and L. major), 2.5 ns = 25 frames. For 10 ps intervals (L. infantum WT), 2.5 ns = 250 frames.

In [ ]:
# Discard first 2.5 ns of equilibration
# L. infantum WT was saved every 10 ps -> 250 frames = 2.5 ns
# Others were saved every 100 ps -> 25 frames = 2.5 ns

wt_eq = wt_protein[250:]
mut_eq = mut_protein[25:]
lmaj_wt_eq = lmaj_wt_protein[25:]
lmaj_mut_eq = lmaj_mut_protein[25:]

print("After removing first 2.5 ns:")
print(f"L. infantum WT:  {wt_eq.n_frames} frames")
print(f"L. infantum MUT: {mut_eq.n_frames} frames")
print(f"L. major WT:     {lmaj_wt_eq.n_frames} frames")
print(f"L. major MUT:    {lmaj_mut_eq.n_frames} frames")

## 5. Calculate RMSD (Whole Protein and Regional)

In [ ]:
# Whole protein RMSD (all frames, for plotting)
wt_rmsd = md.rmsd(wt_protein, wt_protein, 0) * 10
mut_rmsd = md.rmsd(mut_protein, mut_protein, 0) * 10
lmaj_wt_rmsd = md.rmsd(lmaj_wt_protein, lmaj_wt_protein, 0) * 10
lmaj_mut_rmsd = md.rmsd(lmaj_mut_protein, lmaj_mut_protein, 0) * 10

# Post-equilibration RMSD
wt_rmsd_eq = md.rmsd(wt_eq, wt_eq, 0) * 10
mut_rmsd_eq = md.rmsd(mut_eq, mut_eq, 0) * 10
lmaj_wt_rmsd_eq = md.rmsd(lmaj_wt_eq, lmaj_wt_eq, 0) * 10
lmaj_mut_rmsd_eq = md.rmsd(lmaj_mut_eq, lmaj_mut_eq, 0) * 10

print("Whole-protein RMSD (post-equilibration):")
print(f"L. infantum WT:  {np.mean(wt_rmsd_eq):.2f} +/- {np.std(wt_rmsd_eq):.2f} A")
print(f"L. infantum MUT: {np.mean(mut_rmsd_eq):.2f} +/- {np.std(mut_rmsd_eq):.2f} A")
print(f"L. major WT:     {np.mean(lmaj_wt_rmsd_eq):.2f} +/- {np.std(lmaj_wt_rmsd_eq):.2f} A")
print(f"L. major MUT:    {np.mean(lmaj_mut_rmsd_eq):.2f} +/- {np.std(lmaj_mut_rmsd_eq):.2f} A")

In [ ]:
# Regional RMSD (post-equilibration)
regions = {
    'Hinge (40-51)':          'name CA and resid 39 to 50',
    'N-terminal (1-39)':      'name CA and resid 0 to 38',
    'N-term+Hinge (1-55)':    'name CA and resid 0 to 54',
    'Catalytic core (52-425)': 'name CA and resid 51 to 424'
}

region_rmsd = {}
for name, selection in regions.items():
    region_rmsd[name] = {
        'linf_wt':  md.rmsd(wt_eq, wt_eq, 0, atom_indices=wt_eq.topology.select(selection)) * 10,
        'linf_mut': md.rmsd(mut_eq, mut_eq, 0, atom_indices=mut_eq.topology.select(selection)) * 10,
        'lmaj_wt':  md.rmsd(lmaj_wt_eq, lmaj_wt_eq, 0, atom_indices=lmaj_wt_eq.topology.select(selection)) * 10,
        'lmaj_mut': md.rmsd(lmaj_mut_eq, lmaj_mut_eq, 0, atom_indices=lmaj_mut_eq.topology.select(selection)) * 10
    }

print(f"{'Region':<30}{'L.inf WT':>10}{'L.inf MUT':>12}{'L.maj WT':>12}{'L.maj MUT':>12}")
print("-" * 76)
for name in regions.keys():
    r = region_rmsd[name]
    print(f"{name:<30}{np.mean(r['linf_wt']):>10.2f}{np.mean(r['linf_mut']):>12.2f}{np.mean(r['lmaj_wt']):>12.2f}{np.mean(r['lmaj_mut']):>12.2f}")

## 6. Calculate Per-Residue RMSF

In [ ]:
# Superpose all frames on the first frame (post-equilibration)
wt_aligned = wt_eq.superpose(wt_eq, 0)
mut_aligned = mut_eq.superpose(mut_eq, 0)
lmaj_wt_aligned = lmaj_wt_eq.superpose(lmaj_wt_eq, 0)
lmaj_mut_aligned = lmaj_mut_eq.superpose(lmaj_mut_eq, 0)

# Calculate RMSF per atom, then extract CA atoms only for per-residue values
wt_rmsf_ca  = (md.rmsf(wt_aligned, wt_aligned, 0) * 10)[wt_eq.topology.select('name CA')]
mut_rmsf_ca = (md.rmsf(mut_aligned, mut_aligned, 0) * 10)[mut_eq.topology.select('name CA')]
lmaj_wt_rmsf_ca  = (md.rmsf(lmaj_wt_aligned, lmaj_wt_aligned, 0) * 10)[lmaj_wt_eq.topology.select('name CA')]
lmaj_mut_rmsf_ca = (md.rmsf(lmaj_mut_aligned, lmaj_mut_aligned, 0) * 10)[lmaj_mut_eq.topology.select('name CA')]

print(f"L. infantum WT RMSF:  {len(wt_rmsf_ca)} residues")
print(f"L. infantum MUT RMSF: {len(mut_rmsf_ca)} residues")
print(f"L. major WT RMSF:     {len(lmaj_wt_rmsf_ca)} residues")
print(f"L. major MUT RMSF:    {len(lmaj_mut_rmsf_ca)} residues")

## 7. Plot RMSD Time Series

In [ ]:
# L. infantum RMSD
wt_time = np.linspace(0, 25, len(wt_rmsd))
mut_time = np.linspace(0, 25, len(mut_rmsd))

plt.figure(figsize=(10, 5))
plt.plot(wt_time, wt_rmsd, color='blue', alpha=0.7, label='Wild Type')
plt.plot(mut_time, mut_rmsd, color='red', alpha=0.7, label='Mutant')
plt.axvline(x=2.5, color='gray', linestyle='--', alpha=0.5, label='Equilibration cutoff')
plt.xlabel('Time (ns)')
plt.ylabel('RMSD (\u00c5)')
plt.title('L. infantum NTPDase1 - Whole Protein RMSD')
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_folder}/Linfantum_RMSD.png", dpi=300)
plt.show()

# L. major RMSD
lmaj_wt_time = np.linspace(0, 25, len(lmaj_wt_rmsd))
lmaj_mut_time = np.linspace(0, 25, len(lmaj_mut_rmsd))

plt.figure(figsize=(10, 5))
plt.plot(lmaj_wt_time, lmaj_wt_rmsd, color='blue', alpha=0.7, label='Wild Type')
plt.plot(lmaj_mut_time, lmaj_mut_rmsd, color='red', alpha=0.7, label='Mutant')
plt.axvline(x=2.5, color='gray', linestyle='--', alpha=0.5, label='Equilibration cutoff')
plt.xlabel('Time (ns)')
plt.ylabel('RMSD (\u00c5)')
plt.title('L. major NTPDase1 - Whole Protein RMSD')
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_folder}/Lmajor_RMSD.png", dpi=300)
plt.show()

## 8. Plot Per-Residue RMSF

In [ ]:
# L. infantum RMSF
residues_wt = np.arange(1, len(wt_rmsf_ca) + 1)
residues_mut = np.arange(1, len(mut_rmsf_ca) + 1)

plt.figure(figsize=(14, 5))
plt.plot(residues_wt, wt_rmsf_ca, color='blue', alpha=0.7, label='Wild Type')
plt.plot(residues_mut, mut_rmsf_ca, color='red', alpha=0.7, label='Mutant')
plt.axvspan(40, 51, alpha=0.2, color='green', label='Hinge Region (40-51)')
plt.xlabel('Residue Number')
plt.ylabel('RMSF (\u00c5)')
plt.title('L. infantum NTPDase1 - Per-Residue Ca RMSF')
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_folder}/Linfantum_RMSF.png", dpi=300)
plt.show()

# L. major RMSF
residues_wt_maj = np.arange(1, len(lmaj_wt_rmsf_ca) + 1)
residues_mut_maj = np.arange(1, len(lmaj_mut_rmsf_ca) + 1)

plt.figure(figsize=(14, 5))
plt.plot(residues_wt_maj, lmaj_wt_rmsf_ca, color='blue', alpha=0.7, label='Wild Type')
plt.plot(residues_mut_maj, lmaj_mut_rmsf_ca, color='red', alpha=0.7, label='Mutant')
plt.axvspan(40, 51, alpha=0.2, color='green', label='Hinge Region (40-51)')
plt.xlabel('Residue Number')
plt.ylabel('RMSF (\u00c5)')
plt.title('L. major NTPDase1 - Per-Residue Ca RMSF')
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_folder}/Lmajor_RMSF.png", dpi=300)
plt.show()

## 9. Statistical Analysis

**RMSD** — 10-block averaged independent t-test to account for temporal correlation between frames.

**RMSF** — Paired t-test comparing per-residue values between wild-type and mutant within each region.

In [ ]:
n_blocks = 10

def sig_marker(p):
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def block_ttest(wt_data, mut_data, n_blocks=10):
    wt_blocks = np.array_split(wt_data, n_blocks)
    mut_blocks = np.array_split(mut_data, n_blocks)
    wt_means = [np.mean(b) for b in wt_blocks]
    mut_means = [np.mean(b) for b in mut_blocks]
    return stats.ttest_ind(wt_means, mut_means)

print("=" * 70)
print("RMSD STATISTICAL TESTS (10-block t-test, post-equilibration)")
print("=" * 70)
print(f"{'Region':<30}{'L. infantum':>20}{'L. major':>20}")
print("-" * 70)

# Whole protein
t1, p1 = block_ttest(wt_rmsd_eq, mut_rmsd_eq)
t2, p2 = block_ttest(lmaj_wt_rmsd_eq, lmaj_mut_rmsd_eq)
print(f"{'Whole protein':<30}p={p1:.4f} {sig_marker(p1):>4}  p={p2:.4f} {sig_marker(p2):>4}")

# Regional
for name in regions.keys():
    r = region_rmsd[name]
    t1, p1 = block_ttest(r['linf_wt'], r['linf_mut'])
    t2, p2 = block_ttest(r['lmaj_wt'], r['lmaj_mut'])
    print(f"{name:<30}p={p1:.4f} {sig_marker(p1):>4}  p={p2:.4f} {sig_marker(p2):>4}")

In [ ]:
print("=" * 70)
print("RMSF STATISTICAL TESTS (paired t-test, post-equilibration)")
print("=" * 70)
print(f"{'Region':<30}{'L. infantum':>20}{'L. major':>20}")
print("-" * 70)

rmsf_regions = {
    'Hinge (40-51)':         (wt_rmsf_ca[39:51],  mut_rmsf_ca[39:51],  lmaj_wt_rmsf_ca[39:51],  lmaj_mut_rmsf_ca[39:51]),
    'N-terminal (1-39)':     (wt_rmsf_ca[0:39],   mut_rmsf_ca[0:39],   lmaj_wt_rmsf_ca[0:39],   lmaj_mut_rmsf_ca[0:39]),
    'N-term+Hinge (1-55)':   (wt_rmsf_ca[0:55],   mut_rmsf_ca[0:55],   lmaj_wt_rmsf_ca[0:55],   lmaj_mut_rmsf_ca[0:55]),
    'Catalytic core (52-425)': (wt_rmsf_ca[51:], mut_rmsf_ca[51:], lmaj_wt_rmsf_ca[51:], lmaj_mut_rmsf_ca[51:])
}

for name, (wt_d, mut_d, lmaj_wt_d, lmaj_mut_d) in rmsf_regions.items():
    min_len1 = min(len(wt_d), len(mut_d))
    t1, p1 = stats.ttest_rel(wt_d[:min_len1], mut_d[:min_len1])

    min_len2 = min(len(lmaj_wt_d), len(lmaj_mut_d))
    t2, p2 = stats.ttest_rel(lmaj_wt_d[:min_len2], lmaj_mut_d[:min_len2])

    print(f"{name:<30}p={p1:.4f} {sig_marker(p1):>4}  p={p2:.4f} {sig_marker(p2):>4}")

## 10. Percentage Change (Wild Type -> Mutant)

In [ ]:
def pct_change(wt, mut):
    return ((np.mean(mut) - np.mean(wt)) / np.mean(wt)) * 100

print("=" * 60)
print("PERCENTAGE CHANGE (WT -> MUT), post-equilibration")
print("=" * 60)
print(f"{'Region':<30}{'L. infantum':>15}{'L. major':>15}")
print("-" * 60)

# RMSD % changes
print("\nRMSD")
print(f"{'  Whole protein':<30}{pct_change(wt_rmsd_eq, mut_rmsd_eq):>+14.1f}%{pct_change(lmaj_wt_rmsd_eq, lmaj_mut_rmsd_eq):>+14.1f}%")
for name in regions.keys():
    r = region_rmsd[name]
    print(f"{'  ' + name:<30}{pct_change(r['linf_wt'], r['linf_mut']):>+14.1f}%{pct_change(r['lmaj_wt'], r['lmaj_mut']):>+14.1f}%")

# RMSF % changes
print("\nRMSF")
for name, (wt_d, mut_d, lmaj_wt_d, lmaj_mut_d) in rmsf_regions.items():
    print(f"{'  ' + name:<30}{pct_change(wt_d, mut_d):>+14.1f}%{pct_change(lmaj_wt_d, lmaj_mut_d):>+14.1f}%")

## 11. Export Data to CSV

Export raw RMSD time series and per-residue RMSF values for use outside the notebook (e.g. supplementary tables, external plotting).

In [ ]:
# RMSF per residue
pd.DataFrame({
    'Residue': np.arange(1, len(wt_rmsf_ca) + 1),
    'WT_RMSF': wt_rmsf_ca,
    'MUT_RMSF': mut_rmsf_ca[:len(wt_rmsf_ca)]
}).to_csv(f"{output_folder}/Linfantum_RMSF_data.csv", index=False)

pd.DataFrame({
    'Residue': np.arange(1, len(lmaj_wt_rmsf_ca) + 1),
    'WT_RMSF': lmaj_wt_rmsf_ca,
    'MUT_RMSF': lmaj_mut_rmsf_ca[:len(lmaj_wt_rmsf_ca)]
}).to_csv(f"{output_folder}/Lmajor_RMSF_data.csv", index=False)

# RMSD time series
pd.DataFrame({'Time_ns': wt_time,  'WT_RMSD': wt_rmsd}).to_csv(f"{output_folder}/Linfantum_WT_RMSD_data.csv", index=False)
pd.DataFrame({'Time_ns': mut_time, 'MUT_RMSD': mut_rmsd}).to_csv(f"{output_folder}/Linfantum_MUT_RMSD_data.csv", index=False)
pd.DataFrame({'Time_ns': lmaj_wt_time,  'WT_RMSD': lmaj_wt_rmsd}).to_csv(f"{output_folder}/Lmajor_WT_RMSD_data.csv", index=False)
pd.DataFrame({'Time_ns': lmaj_mut_time, 'MUT_RMSD': lmaj_mut_rmsd}).to_csv(f"{output_folder}/Lmajor_MUT_RMSD_data.csv", index=False)

print("All data exported to CSV in results folder")